# Attention by hand: the first CPU increment

This is a transparent numerical lesson, not a trained language model or a microscopy result. Read a section, make a prediction, then execute its cell. Keep `attention.py` in the same directory. No datasets or checkpoints are required.

**Question:** how does one receiving token gather content from source tokens?

**Contract:** explain the mechanism, trace every axis, calculate one row, inspect gradients, break an access rule, and save evidence. Reading and passing assertions do not automatically mean the module is independently explained.

In [ ]:
import json, math, platform, sys
from pathlib import Path
import torch
from attention import scaled_dot_product_attention, analytic_attention_backward, projection_backward, multihead_self_attention, run_checks
print({'python': sys.version.split()[0], 'torch': torch.__version__, 'device': 'cpu'})
torch.manual_seed(17)
dtype = torch.float64


## 1. Distinguish compatibility from content

Queries Q and keys K have the same compatibility width d. Values V can have a different content width. For one head:

S = Q K-transpose / sqrt(d), A = softmax across keys(S), O = A V.

Q describes a receiver's learned compatibility features. K describes a source's compatibility features. V supplies the source content. These are learned projections in a model, rather than literal questions and answers.

For q=[1,0], keys [[1,0],[0,1]], and d=2, calculate scores [1/sqrt(2),0]. Subtract the row maximum before exponentiating. Predict the two weights before running. They should sum to one.

In [ ]:
q = torch.tensor([[[1., 0.]]], dtype=dtype)     # [B=1,T_query=1,d=2]
k = torch.tensor([[[1., 0.], [0., 1.]]], dtype=dtype) # [1,T_key=2,2]
v = torch.tensor([[[2., 0.], [0., 4.]]], dtype=dtype) # [1,2,d_value=2]
out, weights = scaled_dot_product_attention(q, k, v)
print('weights:', weights.tolist())
print('output:', out.tolist())
print('shapes:', tuple(weights.shape), tuple(out.shape))
torch.testing.assert_close(weights.sum(-1), torch.ones(1,1,dtype=dtype))
changed_out, changed_weights = scaled_dot_product_attention(q, k, v * 2)
torch.testing.assert_close(weights, changed_weights)
torch.testing.assert_close(changed_out, out * 2)


## 2. Trace gradients through both paths

Let G be the gradient arriving at O. The value path has dV=A-transpose G. The weight path has dA=G V-transpose. For each row, dS=A * (dA - sum(A*dA)). Finally dQ=dS K/sqrt(d) and dK=dS-transpose Q/sqrt(d).

The row sum in the softmax derivative is over source keys. A fixed blocking mask is not a learned parameter. A learned additive position bias would receive a gradient.

Compare analytical Q/K/V gradients with automatic differentiation and a central finite difference. A tiny epsilon is not always better: rounding error dominates at sufficiently small steps.

In [ ]:
qg = torch.randn(1,3,2,dtype=dtype,requires_grad=True)
kg = torch.randn(1,3,2,dtype=dtype,requires_grad=True)
vg = torch.randn(1,3,4,dtype=dtype,requires_grad=True)
causal3 = torch.ones(3,3,dtype=torch.bool).tril()
y, a = scaled_dot_product_attention(qg,kg,vg,causal3)
g = torch.randn_like(y)
(y*g).sum().backward()
manual = analytic_attention_backward(qg.detach(),kg.detach(),vg.detach(),a.detach(),g,causal3)
for name, analytical, automatic in zip(['Q','K','V'],manual,[qg.grad,kg.grad,vg.grad]):
    torch.testing.assert_close(analytical,automatic,rtol=1e-9,atol=1e-10)
    print(name, 'maximum gradient difference:',float((analytical-automatic).abs().max()))
epsilon=1e-6
qp,qm=qg.detach().clone(),qg.detach().clone()
qp[0,1,0]+=epsilon
qm[0,1,0]-=epsilon
lp=(scaled_dot_product_attention(qp,kg.detach(),vg.detach(),causal3)[0]*g).sum()
lm=(scaled_dot_product_attention(qm,kg.detach(),vg.detach(),causal3)[0]*g).sum()
finite_difference=(lp-lm)/(2*epsilon)
torch.testing.assert_close(finite_difference,qg.grad[0,1,0],rtol=1e-5,atol=1e-7)
print('finite-difference error:',float((finite_difference-qg.grad[0,1,0]).abs()))


## 3. Information access is testable

Our mask convention is Boolean True = allowed. Causality blocks future keys before softmax. Padding blocks artificial source elements; it does not automatically zero padded query outputs or remove padded loss targets.

Predict what happens to earlier outputs when only the last input changes. Then remove the causal mask and observe why training on leaked targets can be misleading. We define fully blocked rows to have zero weights and output, instead of an undefined all-negative-infinity softmax.

In [ ]:
x=torch.randn(1,4,4,dtype=dtype)
changed=x.clone(); changed[:,-1]+=20
causal4=torch.ones(4,4,dtype=torch.bool).tril()
earlier=scaled_dot_product_attention(x,x,x,causal4)[0][:,:-1]
earlier_changed=scaled_dot_product_attention(changed,changed,changed,causal4)[0][:,:-1]
torch.testing.assert_close(earlier,earlier_changed,rtol=0,atol=0)
print('causal earlier difference:',float((earlier-earlier_changed).abs().max()))
leaked=scaled_dot_product_attention(x,x,x)[0][:,:-1]
leaked_changed=scaled_dot_product_attention(changed,changed,changed)[0][:,:-1]
print('unmasked earlier difference:',float((leaked-leaked_changed).abs().max()))
valid=torch.tensor([True,True,False,False])[None,None,:]
padded_changed=x.clone(); padded_changed[:,2:]+=50
p1=scaled_dot_product_attention(x,x,x,valid)[0][:,:2]
p2=scaled_dot_product_attention(padded_changed,padded_changed,padded_changed,valid)[0][:,:2]
torch.testing.assert_close(p1,p2,rtol=0,atol=0)
blocked=torch.zeros(4,4,dtype=torch.bool)
zero,zero_a=scaled_dot_product_attention(x,x,x,blocked)
assert bool((zero==0).all()) and bool((zero_a==0).all())
print('padding invariant and zero-row policy passed')


## 4. Projections and multiple heads

For input [B,T,D], each [D,D] projection can use every original feature. Reshape the projected representation to [B,T,h,d_head], transpose to [B,h,T,d_head], calculate attention, transpose back, concatenate, and apply W_O. Here D=h*d_head.

Reshape reorganizes values; projection learns a new representation. The heads are not fixed semantic partitions of the original input.

The companion module also contains `projection_backward`. In self-attention, the shared input receives the sum of Q-, K-, and V-path gradients. The full invariant suite compares these with automatic differentiation.

In [ ]:
D=4; heads=2
matrices=[torch.randn(D,D,dtype=dtype)*0.2 for _ in range(4)]
multi,multi_weights=multihead_self_attention(x,*matrices,heads=heads,allowed=causal4)
print('output:',tuple(multi.shape),'attention:',tuple(multi_weights.shape))
assert multi.shape==(1,4,4)
assert multi_weights.shape==(1,2,4,4)


## 5. Save numerical evidence

The suite checks analytical and finite-difference gradients, shared-input projection paths, causal invariance, padding invariance, fully blocked rows, an independent per-head loop, and agreement with PyTorch's reference functional attention at zero dropout.

This is one deterministic CPU numerical run. It is not a training result, statistical replication, a paper reproduction, or a benchmark. Change a mask axis intentionally, explain the failure, then restore it. Record why each invariant is relevant before marking the lesson demonstrated.

In [ ]:
results=run_checks()
print(json.dumps(results,indent=2))
evidence={'lesson':'attention-by-hand','seed':17,'dtype':'float64','device':'cpu','torch_version':torch.__version__,'results':results}
Path('my-attention-evidence.json').write_text(json.dumps(evidence,indent=2),encoding='utf-8')
print('Saved my-attention-evidence.json in the notebook working directory.')


## Next increment

Implement a complete block with explicit normalization, feed-forward transforms, residuals, and clearly separated dropout settings. Then test cached inference and positions. Keep microscopy models and real-data results planned until built and evaluated.

Primary readings: [Attention Is All You Need](https://arxiv.org/abs/1706.03762), [PyTorch autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html), and [functional scaled-dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html). Confirm conventions against the installed runtime.

**Check:** changing only V should change output content while keeping attention weights fixed. Explain why before looking back at Section 1.